# 最終テスト+パフォーマンス計測

##　目的
-全パイプラインがエラーなく動作することを確認
-各ステップの実行時間を計測
-プロジェクト全体の成果を数値でまとめる
-ポートフォリオとしての完成度を最終チェック


In [1]:
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats
import yfinance as yf
import time
from datetime import datetime

print("=" * 60)
print("最終テスト開始")
print(f"実行日時: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

最終テスト開始
実行日時: 2026-04-12 15:45:39


## 1. Extract テスト:データ取得

In [2]:
# Extract
print("\n[1/6] Extract: データ取得")
start = time.time()

nikkei = yf.download('^N225', start='2019-01-01')
nikkei.columns = nikkei.columns.get_level_values(0)

extract_time = time.time() - start
extract_rows = len(nikkei)

print(f"  ✓ {extract_rows} 行取得（{extract_time:.2f}秒）")
print(f"  期間: {nikkei.index[0].strftime('%Y-%m-%d')} 〜 {nikkei.index[-1].strftime('%Y-%m-%d')}")
assert extract_rows > 1000, f"データ不足: {extract_rows} 行"
print(f"  ✓ データ量チェック OK（1000行以上）")


[1/6] Extract: データ取得


[*********************100%***********************]  1 of 1 completed

  ✓ 1772 行取得（8.44秒）
  期間: 2019-01-04 〜 2026-04-10
  ✓ データ量チェック OK（1000行以上）


In [3]:
# Transform
print("\n[2/6] Transform: SQL 加工")
start = time.time()

conn = sqlite3.connect('nikkei225.db')

# Raw
df_sql = nikkei[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df_sql.index = df_sql.index.strftime('%Y-%m-%d')
df_sql.index.name = 'date'
df_sql.to_sql('raw_nikkei225', conn, if_exists='replace')

# Staging
conn.execute("DROP TABLE IF EXISTS stg_nikkei225")
conn.execute("""
    CREATE TABLE stg_nikkei225 AS
    SELECT date,
        ROUND(CAST(Open AS REAL), 2) as open_price,
        ROUND(CAST(High AS REAL), 2) as high_price,
        ROUND(CAST(Low AS REAL), 2) as low_price,
        ROUND(CAST(Close AS REAL), 2) as close_price,
        CAST(Volume AS INTEGER) as volume
    FROM raw_nikkei225
    WHERE Close IS NOT NULL ORDER BY date
""")

# Intermediate
conn.execute("DROP TABLE IF EXISTS int_daily_metrics")
conn.execute("""
    CREATE TABLE int_daily_metrics AS
    SELECT date, close_price, volume,
        ROUND((close_price - LAG(close_price) OVER (ORDER BY date))
            / LAG(close_price) OVER (ORDER BY date) * 100, 4) as daily_return,
        ROUND(AVG(close_price) OVER (
            ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW), 2) as ma_20,
        ROUND(AVG(close_price) OVER (
            ORDER BY date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW), 2) as ma_200
    FROM stg_nikkei225 ORDER BY date
""")
conn.commit()

transform_time = time.time() - start
print(f"  ✓ raw → staging → intermediate 完了（{transform_time:.2f}秒）")


[2/6] Transform: SQL 加工
  ✓ raw → staging → intermediate 完了（0.13秒）


In [4]:
# Quality Check
print("\n[3/6] Quality: データ品質テスト")
start = time.time()

tests = {
    "NULL チェック": "SELECT COUNT(*) FROM stg_nikkei225 WHERE close_price IS NULL",
    "重複チェック": "SELECT COUNT(*) FROM (SELECT date, COUNT(*) c FROM stg_nikkei225 GROUP BY date HAVING c > 1)",
    "正の値": "SELECT COUNT(*) FROM stg_nikkei225 WHERE close_price <= 0",
    "高値≧安値": "SELECT COUNT(*) FROM stg_nikkei225 WHERE high_price < low_price",
    "終値が範囲内": "SELECT COUNT(*) FROM stg_nikkei225 WHERE close_price > high_price OR close_price < low_price",
    "日付フォーマット": "SELECT COUNT(*) FROM stg_nikkei225 WHERE date NOT LIKE '____-__-__'",
}

quality_passed = 0
for test_name, query in tests.items():
    result = pd.read_sql(query, conn).iloc[0, 0]
    status = "✓" if result == 0 else "✗"
    print(f"  {status} {test_name}")
    if result == 0:
        quality_passed += 1

quality_time = time.time() - start
print(f"  結果: {quality_passed}/{len(tests)} 通過（{quality_time:.2f}秒）")
assert quality_passed == len(tests), "品質テスト失敗"


[3/6] Quality: データ品質テスト
  ✓ NULL チェック
  ✓ 重複チェック
  ✓ 正の値
  ✓ 高値≧安値
  ✓ 終値が範囲内
  ✓ 日付フォーマット
  結果: 6/6 通過（0.02秒）


In [5]:
# Analysis
print("\n[4/6] Analysis: 統計分析")
start = time.time()

# シグナル生成
signals = pd.read_sql("""
    SELECT date, close_price, ma_20, ma_200,
        CASE WHEN ma_20 > ma_200 
            AND LAG(ma_20) OVER (ORDER BY date) <= LAG(ma_200) OVER (ORDER BY date)
            THEN 'BUY'
            WHEN ma_20 < ma_200 
            AND LAG(ma_20) OVER (ORDER BY date) >= LAG(ma_200) OVER (ORDER BY date)
            THEN 'SELL'
        END as signal
    FROM int_daily_metrics WHERE ma_200 IS NOT NULL
""", conn)

sig = signals[signals['signal'].notna()]
buys = sig[sig['signal'] == 'BUY'].reset_index(drop=True)
sells = sig[sig['signal'] == 'SELL'].reset_index(drop=True)

returns = []
for _, buy in buys.iterrows():
    future = sells[sells['date'] > buy['date']]
    if len(future) > 0:
        sell = future.iloc[0]
        returns.append((sell['close_price'] - buy['close_price']) / buy['close_price'] * 100)

t_stat, p_value = stats.ttest_1samp(returns, 0)

analysis_time = time.time() - start
print(f"  トレード数: {len(returns)}")
print(f"  勝率: {sum(1 for r in returns if r > 0) / len(returns) * 100:.1f}%")
print(f"  平均リターン: {np.mean(returns):.2f}%")
print(f"  t統計量: {t_stat:.4f}")
print(f"  p値: {p_value:.6f}")
print(f"  ✓ 統計分析完了（{analysis_time:.2f}秒）")


[4/6] Analysis: 統計分析
  トレード数: 12
  勝率: 16.7%
  平均リターン: 1.34%
  t統計量: 0.3335
  p値: 0.745045
  ✓ 統計分析完了（0.05秒）


In [6]:
# Granger 因果性テスト
print("\n[5/6] Granger: 因果推論")
start = time.time()

from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

# 米国10年国債利回りを取得
tnx = yf.download('^TNX', start='2019-01-01')
tnx.columns = tnx.columns.get_level_values(0)

df_causal = pd.DataFrame({
    'nikkei_return': nikkei['Close'].pct_change() * 100,
    'yield_change': tnx['Close'].diff()
}).dropna()

test_data = df_causal[['nikkei_return', 'yield_change']].values
results = grangercausalitytests(test_data, maxlag=5, verbose=False)

significant_lags = []
for lag in range(1, 6):
    p = results[lag][0]['ssr_ftest'][1]
    if p < 0.05:
        significant_lags.append(lag)

granger_time = time.time() - start

if significant_lags:
    print(f"  ✓ 因果関係あり（ラグ {significant_lags}）")
else:
    print(f"  ✗ 有意な因果関係なし")
print(f"  完了（{granger_time:.2f}秒）")


[5/6] Granger: 因果推論


[*********************100%***********************]  1 of 1 completed

  ✓ 因果関係あり（ラグ [1, 2]）
  完了（3.00秒）


In [ ]:
# Final Report
print("\n[6/6] Report: 最終レポート生成")
start = time.time()

# テーブル一覧
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)

report_time = time.time() - start
total_time = extract_time + transform_time + quality_time + analysis_time + granger_time + report_time

conn.close()

# ===== パフォーマンスサマリー =====
print(f"\n{'═' * 60}")
print(f"最終テスト結果")
print(f"{'═' * 60}")

perf_data = {
    'ステップ': ['Extract', 'Transform', 'Quality', 'Analysis', 'Granger', 'Report'],
    '時間（秒）': [extract_time, transform_time, quality_time, analysis_time, granger_time, report_time],
    '状態': ['✓'] * 6
}
perf_df = pd.DataFrame(perf_data)
print(perf_df.to_string(index=False))

print(f"\n総実行時間: {total_time:.2f}秒")
print(f"データ行数: {extract_rows}")
print(f"テーブル数: {len(tables)}")
print(f"品質テスト: {quality_passed}/{len(tests)} 通過")
print(f"{'═' * 60}")